# Zarr Conversion Validation

Quick sanity-check: compare the **original** Zarr V2 store with the **converted** Zarr V3 (GeoZarr) store.

Checks performed:
1. Load both stores and inspect structure
2. Data-type and shape comparison per band / resolution
3. Value statistics (mean, std, min, max, p5, p95) for key bands
4. Side-by-side false-colour RGB visualisation (B04/B03/B02)
5. Pixel-level difference map and histogram


## 1 · Paths & imports

In [ ]:
import pathlib
import warnings

import eopf_geozarr.codecs  # noqa: F401 — registers scale_offset codec with zarr registry
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")

# Edit REPO_ROOT if running from a different directory
REPO_ROOT = pathlib.Path("../../").resolve()

SCENE_STEM = "S2C_MSIL2A_20260427T101021_N0512_R022_T33UWT_20260427T151616"

ORIGINAL_PATH = REPO_ROOT / f"{SCENE_STEM}.zarr"
CONVERTED_PATH = REPO_ROOT / f"{SCENE_STEM}_converted.zarr"

if not ORIGINAL_PATH.exists():
    raise FileNotFoundError(f"Original store not found: {ORIGINAL_PATH}")
if not CONVERTED_PATH.exists():
    raise FileNotFoundError(f"Converted store not found: {CONVERTED_PATH}")

print(f"Original  -> {ORIGINAL_PATH}")
print(f"Converted -> {CONVERTED_PATH}")

Original  -> /Users/lhoupert/DevDS/EOPF/data-pipeline/S2C_MSIL2A_20260427T101021_N0512_R022_T33UWT_20260427T151616.zarr
Converted -> /Users/lhoupert/DevDS/EOPF/data-pipeline/S2C_MSIL2A_20260427T101021_N0512_R022_T33UWT_20260427T151616_converted.zarr


## 2 · Open stores

In [2]:
# Original is Zarr V2
dt_orig = xr.open_datatree(
    str(ORIGINAL_PATH),
    engine="zarr",
    chunks={},
    consolidated=False,
)

# Converted is Zarr V3
dt_conv = xr.open_datatree(
    str(CONVERTED_PATH),
    engine="zarr",
    chunks={},
    consolidated=False,
)

print("=== ORIGINAL tree ===")
print(dt_orig)
print()
print("=== CONVERTED tree ===")
print(dt_conv)

UnknownCodecError: Unknown codec: 'scale_offset'

## 3 · Shape & dtype comparison

In [3]:
RESOLUTIONS = ["r10m", "r20m", "r60m"]
GROUP_PATH = "measurements/reflectance"

rows = []
for res in RESOLUTIONS:
    try:
        ds_o = dt_orig[f"/{GROUP_PATH}/{res}"].to_dataset()
    except KeyError:
        print(f"  [skip] original has no {GROUP_PATH}/{res}")
        continue
    try:
        ds_c = dt_conv[f"/{GROUP_PATH}/{res}"].to_dataset()
    except KeyError:
        print(f"  [skip] converted has no {GROUP_PATH}/{res}")
        continue

    orig_bands = [v for v in ds_o.data_vars if v not in ("x", "y", "spatial_ref")]
    conv_bands = [v for v in ds_c.data_vars if v not in ("x", "y", "spatial_ref")]
    common_bands = sorted(set(orig_bands) & set(conv_bands))

    for band in common_bands:
        o = ds_o[band]
        c = ds_c[band]
        rows.append(
            {
                "resolution": res,
                "band": band,
                "shape_orig": str(o.shape),
                "shape_conv": str(c.shape),
                "dtype_orig": str(o.dtype),
                "dtype_conv": str(c.dtype),
                "shape_match": o.shape == c.shape,
            }
        )

df_shapes = pd.DataFrame(rows)
print(df_shapes.to_string(index=False))

if df_shapes["shape_match"].all():
    print("\n All shapes match")
else:
    print("\n Shape mismatches detected")

NameError: name 'dt_conv' is not defined

## 4 · Value statistics

Computed on the first 512x512 px tile for speed.


In [ ]:
TILE = 512

stat_rows = []
for _, row in df_shapes.iterrows():
    res, band = row["resolution"], row["band"]
    ds_o = dt_orig[f"/{GROUP_PATH}/{res}"].to_dataset()
    ds_c = dt_conv[f"/{GROUP_PATH}/{res}"].to_dataset()

    sdims_o = [d for d in ds_o[band].dims if d not in ("time", "band")]
    sdims_c = [d for d in ds_c[band].dims if d not in ("time", "band")]

    arr_o = ds_o[band].isel({d: slice(0, TILE) for d in sdims_o}).values.astype(float)
    arr_c = ds_c[band].isel({d: slice(0, TILE) for d in sdims_c}).values.astype(float)

    def _stats(a, label):
        flat = a[np.isfinite(a)].ravel()
        if flat.size == 0:
            return {
                f"{label}_{k}": float("nan") for k in ["mean", "std", "min", "p5", "p95", "max"]
            }
        return {
            f"{label}_mean": float(flat.mean()),
            f"{label}_std": float(flat.std()),
            f"{label}_min": float(flat.min()),
            f"{label}_p5": float(np.percentile(flat, 5)),
            f"{label}_p95": float(np.percentile(flat, 95)),
            f"{label}_max": float(flat.max()),
        }

    stat_rows.append(
        {"resolution": res, "band": band, **_stats(arr_o, "orig"), **_stats(arr_c, "conv")}
    )

df_stats = pd.DataFrame(stat_rows)
pd.options.display.float_format = "{:.3f}".format
print(df_stats.to_string(index=False))

## 5 · Mean-ratio check

Ratio ~1.0 = same scale.  Ratio ~0.0001 = one store uses DN, other uses physical reflectance.


In [ ]:
df_ratio = df_stats[["resolution", "band", "orig_mean", "conv_mean"]].copy()
df_ratio["ratio_conv_orig"] = df_ratio["conv_mean"] / df_ratio["orig_mean"]
print(df_ratio.to_string(index=False))

ratios = df_ratio["ratio_conv_orig"].dropna()
if ratios.between(0.8, 1.2).all():
    print("\n Values are on the same scale (ratio within 0.8-1.2)")
else:
    print(f"\n Systematic scale difference detected (median ratio {ratios.median():.4f})")
    print("   This may be expected if one store uses DN and the other physical reflectance.")

## 6 · Visual comparison -- B04/B03/B02 false colour RGB

In [ ]:
VIZ_RES = "r10m"
VIZ_TILE = 512
VIZ_PMIN, VIZ_PMAX = 2, 98

ds_o_10 = dt_orig[f"/{GROUP_PATH}/{VIZ_RES}"].to_dataset()
ds_c_10 = dt_conv[f"/{GROUP_PATH}/{VIZ_RES}"].to_dataset()


def _load_band(ds, band, tile):
    dims = [d for d in ds[band].dims if d not in ("time", "band")]
    return ds[band].isel({d: slice(0, tile) for d in dims}).values.astype(float)


def _rgb(ds, bands, tile):
    channels = [_load_band(ds, b, tile) for b in bands]
    stack = np.stack(channels, axis=-1)
    lo = np.nanpercentile(stack, VIZ_PMIN, axis=(0, 1), keepdims=True)
    hi = np.nanpercentile(stack, VIZ_PMAX, axis=(0, 1), keepdims=True)
    return np.clip((stack - lo) / (hi - lo + 1e-9), 0, 1)


RGB_BANDS = ["b04", "b03", "b02"]
avail_o = [b for b in RGB_BANDS if b in ds_o_10.data_vars]
avail_c = [b for b in RGB_BANDS if b in ds_c_10.data_vars]

if len(avail_o) == 3 and len(avail_c) == 3:
    rgb_orig = _rgb(ds_o_10, RGB_BANDS, VIZ_TILE)
    rgb_conv = _rgb(ds_c_10, RGB_BANDS, VIZ_TILE)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(rgb_orig)
    axes[0].set_title("Original (B04/B03/B02)")
    axes[0].axis("off")
    axes[1].imshow(rgb_conv)
    axes[1].set_title("Converted (B04/B03/B02)")
    axes[1].axis("off")
    fig.suptitle(f"{SCENE_STEM} -- {VIZ_RES} -- first {VIZ_TILE}x{VIZ_TILE} px", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print(f"RGB bands not fully available: orig={avail_o} conv={avail_c}")

## 7 · Pixel-level difference (B02 r10m)

In [ ]:
DIFF_BAND = "b02"
DIFF_TILE = 512

arr_o = _load_band(ds_o_10, DIFF_BAND, DIFF_TILE)
arr_c = _load_band(ds_c_10, DIFF_BAND, DIFF_TILE)

# Auto-scale if a systematic factor exists (e.g. DN vs reflectance)
scale = np.nanmean(arr_o) / (np.nanmean(arr_c) + 1e-9)
if abs(scale - 1.0) > 0.1:
    print(f"  Applying scale factor {scale:.4f} to converted before diff")
    arr_c_sc = arr_c * scale
else:
    arr_c_sc = arr_c

diff = arr_c_sc - arr_o
rel_err = np.abs(diff) / (np.abs(arr_o) + 1e-9)

print(f"B02 {DIFF_TILE}x{DIFF_TILE} px diff stats:")
print(f"  mean  diff :  {diff.mean():.4f}")
print(f"  std   diff :  {diff.std():.4f}")
print(f"  max |diff| :  {np.abs(diff).max():.4f}")
print(f"  mean rel err: {rel_err.mean():.4%}")

vmax = np.percentile(np.abs(diff), 99)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(diff, cmap="RdBu", vmin=-vmax, vmax=vmax)
axes[0].set_title(f"Pixel diff (conv - orig), {DIFF_BAND} r10m")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.03)

axes[1].hist(diff.ravel(), bins=100, color="steelblue", edgecolor="none")
axes[1].set_xlabel("conv - orig")
axes[1].set_ylabel("pixel count")
axes[1].set_title(f"Difference histogram ({DIFF_BAND})")
axes[1].axvline(0, color="red", linewidth=1, linestyle="--")

plt.tight_layout()
plt.show()

## 8 · Coordinate / spatial extent check

In [ ]:
coord_rows = []
for res in RESOLUTIONS:
    try:
        ds_o = dt_orig[f"/{GROUP_PATH}/{res}"].to_dataset()
        ds_c = dt_conv[f"/{GROUP_PATH}/{res}"].to_dataset()
    except KeyError:
        continue
    for coord in ("x", "y"):
        if coord not in ds_o.coords or coord not in ds_c.coords:
            continue
        xo = ds_o[coord].values
        xc = ds_c[coord].values
        coord_rows.append(
            {
                "res": res,
                "coord": coord,
                "orig_min": float(xo.min()),
                "orig_max": float(xo.max()),
                "conv_min": float(xc.min()),
                "conv_max": float(xc.max()),
                "len_orig": len(xo),
                "len_conv": len(xc),
                "len_match": len(xo) == len(xc),
            }
        )

df_coords = pd.DataFrame(coord_rows)
pd.options.display.float_format = "{:.1f}".format
print(df_coords.to_string(index=False))

if df_coords["len_match"].all():
    print("\n Coordinate lengths match for all resolutions")
else:
    print("\n Coordinate length mismatch detected")

## 9 · Converted-only content

Overview groups (r120m, r360m, r720m) and spatial_ref / CRS variable.


In [ ]:
EXTRA_RES = ["r120m", "r360m", "r720m"]
for res in EXTRA_RES:
    key = f"/{GROUP_PATH}/{res}"
    try:
        ds = dt_conv[key].to_dataset()
        bands = [v for v in ds.data_vars if v not in ("x", "y", "spatial_ref")]
        shapes = {b: ds[b].shape for b in bands[:4]}
        print(f"  {key}  bands={bands}  shapes={shapes}")
    except KeyError:
        print(f"   {key} not present")

ds_c_10m = dt_conv[f"/{GROUP_PATH}/r10m"].to_dataset()
if "spatial_ref" in ds_c_10m:
    sr = ds_c_10m["spatial_ref"]
    crs = sr.attrs.get("crs_wkt", sr.attrs.get("spatial_ref", "<not in attrs>"))
    crs_str = str(crs)
    print(f'\n spatial_ref present -- CRS: {crs_str[:80]}{"..." if len(crs_str) > 80 else ""}')
else:
    print("\n spatial_ref missing from converted r10m group")

## 10 · Summary

In [ ]:
print("=" * 60)
print("Conversion validation summary")
print("=" * 60)

checks = [
    ("Shape match (all bands/resolutions)", bool(df_shapes["shape_match"].all())),
    ("Coordinate lengths match", bool(df_coords["len_match"].all())),
    (
        "Value scale consistent (ratio 0.8-1.2)",
        bool((df_stats["conv_mean"] / df_stats["orig_mean"]).dropna().between(0.8, 1.2).all()),
    ),
]

for label, ok in checks:
    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}]  {label}")

print()
if all(ok for _, ok in checks):
    print("All checks passed -- conversion looks good.")
else:
    print("Some checks failed -- investigate highlighted items above.")